In [17]:
import pandas as pd
from pyfaidx import Fasta
from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from Bio.Seq import Seq

In [18]:
celline_arrary_df = pd.read_csv("/lulabdata3/huangkeyun/zhangys/RNA_locator/circExor/sample_preprocessing/circRNA_external_testset/BRCA_celline_circRNA_arrary.csv")
celline_arrary_df

,CircRNA,MDA-MB-231,MCF-7,averange,Chromosome,CircStart,CircEnd,Strand,Spliced seq length,Best transcript,Gene Symbol
0,hsa_circ_0009688,1014.974000,1760.40800,NaN,chr1,10165573,10165802,+,229,ENST00000253251,UBE4B
1,hsa-circRNA8877-1,4161.921000,3812.78700,NaN,chr1,192992973,192998787,-,486,ENST00000367455,UCHL5
2,hsa_circ_0112904,2095.288000,1396.63800,NaN,chr1,25773252,25783322,+,572,ENST00000374343,TMEM57
3,hsa-circRNA8495-17,734.735300,1047.67100,NaN,chr1,47046201,47049001,-,200,ENST00000342571,MKNK1
4,hsa_circ_0011599,6.024141,17.23607,NaN,chr1,36282482,36316654,+,2458,ENST00000373210,AGO4
...,...,...,...,...,...,...,...,...,...,...,...
170335,hsa-circRNA16316-13,7.062097,35.12470,NaN,chrY,15447442,15478273,-,1804,ENST00000331397,UTY
170336,hsa_circ_0009019,5.152691,15.15177,NaN,chrY,14821211,14821476,+,265,ENST00000338981,USP9Y
170337,hsa_circ_0092207,2798.053000,2376.96700,NaN,chrY,2087554,2111271,-,1924,---,-
170338,hsa_circ_0092259,594.068200,1115.36700,NaN,chrY,14958257,14959252,+,496,ENST00000338981,USP9Y


In [19]:
from Bio import SeqIO

# 使用 Bio 库读取 fasta 文件并构建序列字典
fasta_file = "/lulabdata3/huangkeyun/zhangys/RNA_locator/references/circRNA/hsa_circbase_seq.fa"
# 假设 FASTA id 为序列名（如果有管道符分隔，取第一部分）
fasta_dict = {record.id.split('|')[0]: str(record.seq) for record in SeqIO.parse(fasta_file, "fasta")}

# 根据 CircRNA 列匹配并插入到 sequence 列
celline_arrary_df['sequence'] = celline_arrary_df['CircRNA'].map(fasta_dict)

celline_arrary_df

,CircRNA,MDA-MB-231,MCF-7,averange,Chromosome,CircStart,CircEnd,Strand,Spliced seq length,Best transcript,Gene Symbol,sequence
0,hsa_circ_0009688,1014.974000,1760.40800,NaN,chr1,10165573,10165802,+,229,ENST00000253251,UBE4B,TATTCTCCGATTTTAAGGACTTGATTGGCCAGATTTTAATGGAAGT...
1,hsa-circRNA8877-1,4161.921000,3812.78700,NaN,chr1,192992973,192998787,-,486,ENST00000367455,UCHL5,NaN
2,hsa_circ_0112904,2095.288000,1396.63800,NaN,chr1,25773252,25783322,+,572,ENST00000374343,TMEM57,TACATTTTTATACCTGAAATTCCTGGTGGTGTGGGCACTTGTCCTC...
3,hsa-circRNA8495-17,734.735300,1047.67100,NaN,chr1,47046201,47049001,-,200,ENST00000342571,MKNK1,NaN
4,hsa_circ_0011599,6.024141,17.23607,NaN,chr1,36282482,36316654,+,2458,ENST00000373210,AGO4,GACCTCCGGCTAGCCTGTTTCAGCCACCTCGTCGTCCTGGCCTTGG...
...,...,...,...,...,...,...,...,...,...,...,...,...
170335,hsa-circRNA16316-13,7.062097,35.12470,NaN,chrY,15447442,15478273,-,1804,ENST00000331397,UTY,NaN
170336,hsa_circ_0009019,5.152691,15.15177,NaN,chrY,14821211,14821476,+,265,ENST00000338981,USP9Y,GTTATGAAATGGTCTCTGCAAGATGTTTTGTCCTTGAATTGGAAAT...
170337,hsa_circ_0092207,2798.053000,2376.96700,NaN,chrY,2087554,2111271,-,1924,---,-,TGCCTGCTACTCACCCCACGCAGCCTACGCCCAGAGCAAGCTGGCC...
170338,hsa_circ_0092259,594.068200,1115.36700,NaN,chrY,14958257,14959252,+,496,ENST00000338981,USP9Y,GTGTGGCAGAAAAAACACAGCTTCTGAAATTGAATGTACCTGCTAC...


In [20]:
celline_arrary_df['sequence'].isnull().sum()  # 统计缺失值数量

35645

In [21]:
missing_seq_mask = celline_arrary_df['sequence'].isna()
missing_df = celline_arrary_df[missing_seq_mask].copy()

print(f"初始缺失序列的记录数: {len(missing_df)}")

if not missing_df.empty:
    # 确保坐标列为整型
    missing_df['CircStart'] = missing_df['CircStart'].astype(int)
    missing_df['CircEnd'] = missing_df['CircEnd'].astype(int)

    # 读取 bed reference
    bed_df = pd.read_csv('/lulabdata3/huangkeyun/zhangys/RNA_locator/resources/circAtlas/human_bed_v3.0.txt', sep='\t')
    bed_df['Start'] = bed_df['Start'].astype(int)
    bed_df['End'] = bed_df['End'].astype(int)
    
    # 打印前几行比对一下染色体格式是否一致（例如 'chr1' vs '1'）
    print(f"missing_df 染色体格式示例: {missing_df['Chromosome'].iloc[0] if len(missing_df) > 0 else 'N/A'}")
    print(f"bed_df 染色体格式示例: {bed_df['Chro'].iloc[0] if len(bed_df) > 0 else 'N/A'}")

    # 遍历容差范围内所有的精确组合 (-1, 0, 1) 进行匹配
    matched_chunks = []
    for d_start in [-1, 0, 1]:
        for d_end in [-1, 0, 1]:
            temp_missing = missing_df.copy()
            temp_missing['Start_match'] = temp_missing['CircStart'] + d_start
            temp_missing['End_match'] = temp_missing['CircEnd'] + d_end
            
            chunk = pd.merge(
                temp_missing[['CircRNA', 'Chromosome', 'Strand', 'Start_match', 'End_match']], 
                bed_df[['Chro', 'Strand', 'Start', 'End', 'circAltas_ID']], 
                left_on=['Chromosome', 'Strand', 'Start_match', 'End_match'], 
                right_on=['Chro', 'Strand', 'Start', 'End'],
                how='inner'
            )
            matched_chunks.append(chunk)

    if matched_chunks:
        # 合并所有匹配并去重
        matched_df = pd.concat(matched_chunks, ignore_index=True)
        matched_df = matched_df.drop_duplicates(subset=['CircRNA', 'circAltas_ID'])
        
        print(f"坐标匹配成功记录数(去重后): {len(matched_df)}")
        
        if not matched_df.empty:
            # 读取 sequence 文件
            seq_df = pd.read_csv('/lulabdata3/huangkeyun/zhangys/RNA_locator/resources/circAtlas/human_sequence_v3.0', sep=' ', header=None, names=['circAltas_ID', 'Sequence'])
            
            # 通过 circAltas_ID 获取序列
            merged_seq = pd.merge(matched_df, seq_df, on='circAltas_ID', how='left')
            
            # 统计有多少个拿到了非空序列
            valid_seq_count = merged_seq['Sequence'].notna().sum()
            print(f"通过 circAltas_ID 成功拉取到序列的记录数: {valid_seq_count}")
            
            # 聚合匹配到的序列建立映射字典 (若匹配到多个序列，用逗号连接)
            seq_agg_dict = merged_seq.groupby('CircRNA')['Sequence'].apply(lambda x: ','.join(x.dropna().astype(str))).to_dict()
            
            print(f"准备回填的独立 CircRNA 数量: {len(seq_agg_dict)}")

            # 将匹配结果回填到序列为空的行
            # 这里需要注意填充时保留原有的 NaN （避免被空字符串覆盖）
            celline_arrary_df.loc[missing_seq_mask, 'sequence'] = celline_arrary_df.loc[missing_seq_mask, 'CircRNA'].map(seq_agg_dict)
    else:
        print("未执行坐标匹配逻辑（无匹配块）。")

print(f"回填后现在仍然缺失序列的记录数: {celline_arrary_df['sequence'].isna().sum()}")

初始缺失序列的记录数: 35645
missing_df 染色体格式示例: chr1
bed_df 染色体格式示例: KI270742.1
坐标匹配成功记录数(去重后): 173
通过 circAltas_ID 成功拉取到序列的记录数: 173
准备回填的独立 CircRNA 数量: 173
回填后现在仍然缺失序列的记录数: 35472


In [22]:
# 只保留 sequence 非空的行
filtered_df = celline_arrary_df[celline_arrary_df['sequence'].notna()].copy()


# filtered_df.to_csv("./filtered_celline_arrary_df.csv", index=False)
# 过滤 sequence 长度大于 10000 的行
filtered_df = filtered_df[filtered_df['sequence'].str.len() <= 10000]

# 过滤 sequence 中存在 N（忽略大小写）的行
filtered_df = filtered_df[~filtered_df['sequence'].str.contains('N', case=False, na=False)]

In [ ]:
columns_to_save = ["CircRNA", "MDA-MB-231_original", "Spliced seq length", "Sequence"]
filtered_df.to_csv("./filtered_celline_arrary_df.csv", index=False)